# LUMIN Agents (OpenAI + OLMo)

This notebook wires the LUMIN query compiler to two LLM backends:

- OpenAI (via `OPENAI_API_KEY`)
- OLMo 7B Instruct (OpenAI-compatible endpoint via `OLMO_BASE_URL` and `OLMO_API_KEY`)

In [20]:
# Clone the repo into Colab
!git clone https://github.com/anhkos/LUMIN-v2-Search-Agent.git

fatal: destination path 'LUMIN-v2-Search-Agent' already exists and is not an empty directory.


In [ ]:
%pip install -r "/content/LUMIN-v2-Search-Agent/requirements.txt"

In [22]:
import os
import sys

repo_root = "/content/LUMIN-v2-Search-Agent"
sys.path.insert(0, os.path.join(repo_root, "src"))
print("repo_root:", repo_root)

repo_root: /content/LUMIN-v2-Search-Agent


In [ ]:
import json
import os
import sys
import ast
from dotenv import load_dotenv # type: ignore
from openai import OpenAI  # type: ignore

repo_root = "/content/LUMIN-v2-Search-Agent"
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from logic_engine import NeuroSymbolicSolver

load_dotenv()
ontology_path = os.path.join(repo_root, "data", "ontology.json")
solver = NeuroSymbolicSolver(ontology_path=ontology_path)

In [64]:
import math

EMBED_MODEL = "text-embedding-3-small"
EMBED_CACHE_PATH = os.path.join(repo_root, "data", "ontology_embeddings.json")

# Optional manual synonyms for deterministic mapping
SYNONYMS = {
    # "thermal data": "Thermal Emission",
}

def _normalize_text(text):
    return " ".join(text.lower().strip().split())

def _cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return dot / (norm_a * norm_b)

def _load_embeddings_cache():
    if os.path.isfile(EMBED_CACHE_PATH):
        with open(EMBED_CACHE_PATH, "r") as f:
            data = json.load(f)
        if data.get("model") == EMBED_MODEL:
            return data
    return None

def _save_embeddings_cache(terms, vectors):
    data = {"model": EMBED_MODEL, "terms": terms, "vectors": vectors}
    os.makedirs(os.path.dirname(EMBED_CACHE_PATH), exist_ok=True)
    with open(EMBED_CACHE_PATH, "w") as f:
        json.dump(data, f)
    return data

def build_embeddings_cache(force=False):
    if not force:
        cached = _load_embeddings_cache()
        if cached:
            return cached

    client = OpenAI()
    terms = list(solver.ontology.keys())
    resp = client.embeddings.create(model=EMBED_MODEL, input=terms)
    vectors = [item.embedding for item in resp.data]
    return _save_embeddings_cache(terms, vectors)

def select_ontology(term, top_k=5, threshold=0.78):
    term_norm = _normalize_text(term)
    # Exact match
    if term in solver.ontology:
        return {"ok": True, "term": term, "score": 1.0, "candidates": []}

    # Synonym match
    synonym = SYNONYMS.get(term_norm)
    if synonym and synonym in solver.ontology:
        return {"ok": True, "term": synonym, "score": 1.0, "candidates": []}

    # Embedding match
    cache = build_embeddings_cache()
    client = OpenAI()
    query_vec = client.embeddings.create(model=EMBED_MODEL, input=[term]).data[0].embedding
    scored = []
    for t, v in zip(cache["terms"], cache["vectors"]):
        scored.append((t, _cosine_similarity(query_vec, v)))
    scored.sort(key=lambda x: x[1], reverse=True)
    candidates = [{"term": t, "score": round(s, 4)} for t, s in scored[:top_k]]
    best_term, best_score = scored[0]
    if best_score >= threshold:
        return {"ok": True, "term": best_term, "score": round(best_score, 4), "candidates": candidates}
    return {"ok": False, "term": None, "score": round(best_score, 4), "candidates": candidates}

In [70]:
SYSTEM_PROMPT = """
You are the LUMIN Query Compiler for NASA PDS.
Your goal is to translate Natural Language into a Logic S-Expression.

### ONTOLOGY (Valid Concepts):
{ontology_keys}

### OPERATIONS:
1. INTERSECT(A, B) -> Returns overlap.
2. UNION(A, B) -> Returns combination.
3. DIFFERENCE(Base, Subtract) -> Removes constraints.

### RULES:
- Output ONLY the tuple plan. No markdown, no explanation, no code fences.
- Use Concept Names EXACTLY as listed in the Ontology when you are confident.
- If unsure, output a plain-language concept; a resolver will map it to ontology terms.
- If the user asks for a specific time/value not in ontology, map it to the closest concept available in the ontology.

### EXAMPLES:
User: "Southern summer images"
Output: "Southern Summer"

User: "Southern summer but not polar regions"
Output: ('DIFFERENCE', 'Southern Summer', 'Polar Regions')
"""

In [ ]:
import re

def _build_prompt():
    ontology_keys = ', '.join(solver.ontology.keys())
    return SYSTEM_PROMPT.format(ontology_keys=ontology_keys)

def _strip_fences(text):
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        lines = text.splitlines()
        # Drop opening fence (possibly with language) and closing fence
        if len(lines) >= 2:
            return "\n".join(lines[1:-1]).strip()
    return text

def _function_to_tuple(text):
    pattern = r"\b(INTERSECT|UNION|DIFFERENCE)\s*\("
    return re.sub(pattern, r"('\1', ", text)

def _parse_simple_op(text):
    # Handles forms like AND(METADATA, Dust Storm Season)
    cleaned = text.strip()
    if cleaned.startswith("(") and cleaned.endswith(")"):
        cleaned = cleaned[1:-1].strip()
    m = re.match(r"^(AND|INTERSECT|UNION|DIFFERENCE)\s*\((.*)\)\s*$", cleaned, re.IGNORECASE)
    if not m:
        return None
    op = m.group(1).upper()
    if op == "AND":
        op = "INTERSECT"
    args = [a.strip() for a in m.group(2).split(",", 1)]
    if len(args) != 2:
        return None
    return (op, args[0], args[1])

def _parse_plan(raw_plan):
    # Safer than eval for tuple parsing
    cleaned = _strip_fences(raw_plan)
    if cleaned in solver.ontology:
        return cleaned
    # If it looks like a plain phrase, treat it as a term and resolve later
    if re.fullmatch(r"[A-Za-z][A-Za-z\s_-]*", cleaned):
        return cleaned
    # Simple AND/INTERSECT style without quotes
    simple = _parse_simple_op(cleaned)
    if simple is not None:
        return simple
    try:
        return ast.literal_eval(cleaned)
    except Exception:
        converted = _function_to_tuple(cleaned)
        return ast.literal_eval(converted)

def _field_for_term(term):
    if isinstance(term, str):
        return solver.ontology.get(term, {}).get('field')
    return None

def _resolve_terms(plan, unresolved):
    if isinstance(plan, str):
        if plan in solver.ontology:
            return plan
        match = select_ontology(plan)
        if match.get("ok") and match.get("term"):
            return match["term"]
        unresolved.append({"term": plan, "candidates": match.get("candidates", [])})
        return plan
    if isinstance(plan, tuple):
        return tuple(_resolve_terms(item, unresolved) for item in plan)
    return plan

def _normalize_plan(plan):
    if isinstance(plan, str):
        return plan
    if isinstance(plan, tuple):
        if len(plan) == 2:
            # Treat a 2-tuple as a UNION of two concepts
            return ('UNION', _normalize_plan(plan[0]), _normalize_plan(plan[1]))
        if len(plan) != 3:
            return plan
        op, arg1, arg2 = plan
        arg1 = _normalize_plan(arg1)
        arg2 = _normalize_plan(arg2)
        if op == 'INTERSECT':
            field1 = _field_for_term(arg1)
            field2 = _field_for_term(arg2)
            if field1 and field2 and field1 != field2:
                return ('UNION', arg1, arg2)
        return (op, arg1, arg2)
    return plan

def _exec_plan(plan):
    try:
        result = solver.execute_plan(plan)
        return True, result, None
    except Exception as exc:
        return False, None, str(exc)

def run_agent(user_query, client, model):
    prompt = _build_prompt()
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': prompt},
            {'role': 'user', 'content': user_query}
        ],
        temperature=0.0,
    )
    raw_plan = response.choices[0].message.content.strip()
    try:
        parsed_plan = _parse_plan(raw_plan)
        unresolved = []
        resolved_plan = _resolve_terms(parsed_plan, unresolved)
        normalized_plan = _normalize_plan(resolved_plan)
    except Exception as exc:
        diagnostics = {
            "error": f"parse_error: {exc}",
            "raw_plan": raw_plan
        }
        return raw_plan, diagnostics

    ok, result, error = _exec_plan(normalized_plan)
    if ok:
        if unresolved:
            return raw_plan, {"result": result, "unresolved": unresolved}
        return raw_plan, result

    # Provide failure diagnostics while preserving the raw model output
    diagnostics = {
        "error": error,
        "raw_plan": raw_plan,
        "parsed_plan": parsed_plan,
        "resolved_plan": resolved_plan,
        "normalized_plan": normalized_plan,
        "unresolved": unresolved
    }
    return raw_plan, diagnostics

In [32]:
def openai_agent(user_query, model='gpt-4o'):
    client = OpenAI()
    return run_agent(user_query, client, model)

def olmo_agent(user_query, model='allenai/olmo-3-7b-instruct'):
    api_key = os.getenv('OPENROUTER_API_KEY')
    if not api_key:
        raise ValueError('Missing OPENROUTER_API_KEY env var')
    
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=api_key,
        default_headers={
            'HTTP-Referer': os.getenv('OPENROUTER_SITE_URL', ''),
            'X-Title': os.getenv('OPENROUTER_SITE_NAME', '')
        }
    )
    
    return run_agent(user_query, client, model)

## Example Usage

In [35]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter OPENROUTER_API_KEY: ")

In [72]:
query = 'I need thermal data taken at midnight'

# OpenAI
openai_plan, openai_result = openai_agent(query)
print('OpenAI Plan:', openai_plan)
print('OpenAI Result:', json.dumps(openai_result, indent=2))

# OLMo (requires OLMO_BASE_URL to be set)
olmo_plan, olmo_result = olmo_agent(query)
print('OLMo Plan:', olmo_plan)
print('OLMo Result:', json.dumps(olmo_result, indent=2))

OpenAI Plan: Midnight
OpenAI Result: {
  "type": "cyclic_range",
  "field": "local_true_solar_time",
  "min": 23.0,
  "max": 1.0,
  "range_max": 24.0,
  "description": "Requires cyclic logic handling across the 24h boundary"
}
OLMo Plan: Midnight
OLMo Result: {
  "type": "cyclic_range",
  "field": "local_true_solar_time",
  "min": 23.0,
  "max": 1.0,
  "range_max": 24.0,
  "description": "Requires cyclic logic handling across the 24h boundary"
}
